In [2]:
# ============================================================
# 14. VALIDATE DIFFERENT FARMING CASES
# ============================================================

# Step 1: Import libraries
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score


# ============================================================
# Step 2: Load ACTUAL Dataset
# ============================================================

file_path = "/content/smartagri_crop_yield_dataset.csv"

df = pd.read_csv(file_path)

print("Dataset loaded successfully!")
print("Dataset shape:", df.shape)

print("\nAvailable Crops:")
print(df["Crop"].unique())


# ============================================================
# Step 3: Prepare Data
# ============================================================

features = [
    "Crop",
    "Soil_Type",
    "Soil_pH",
    "Nitrogen",
    "Phosphorus",
    "Potassium",
    "Temperature",
    "Rainfall",
    "Humidity"
]

target = "Yield"

X = df[features]
y = df[target]


# ============================================================
# Step 4: Encode Categorical Data
# ============================================================

categorical_features = [
    "Crop",
    "Soil_Type"
]

preprocessor = ColumnTransformer(
    transformers=[
        (
            "categorical",
            OneHotEncoder(handle_unknown="ignore"),
            categorical_features
        )
    ],
    remainder="passthrough"
)


# ============================================================
# Step 5: Create Model
# ============================================================

model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "regressor",
            RandomForestRegressor(
                n_estimators=100,
                random_state=42
            )
        )
    ]
)


# ============================================================
# Step 6: Train and Test
# ============================================================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

model.fit(X_train, y_train)

print("\nModel trained successfully!")


# ============================================================
# Step 7: Forecast Accuracy
# ============================================================

y_pred = model.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print("\n========== MODEL ACCURACY ==========")
print("MAE  :", round(mae, 2))
print("RMSE :", round(rmse, 2))
print("R2   :", round(r2, 2))


# ============================================================
# Step 8: Function to Generate Advisory
# ============================================================

average_yield = df["Yield"].mean()


def get_yield_status(predicted_yield):

    if predicted_yield >= average_yield * 1.10:
        return "High"

    elif predicted_yield >= average_yield * 0.90:
        return "Medium"

    else:
        return "Low"


def get_advisory(row, predicted_yield):

    advice = []

    # Yield condition
    status = get_yield_status(predicted_yield)

    if status == "High":
        advice.append("Expected yield is high.")

    elif status == "Medium":
        advice.append("Expected yield is medium.")

    else:
        advice.append("Expected yield is low. Crop management should be monitored.")


    # Soil pH
    if row["Soil_pH"] < 6:
        advice.append("Soil pH is low. Suitable soil correction may be required.")

    elif row["Soil_pH"] > 7.5:
        advice.append("Soil pH is high. Suitable soil management may be required.")

    else:
        advice.append("Soil pH is within the selected normal range.")


    # Nitrogen
    if row["Nitrogen"] < 50:
        advice.append("Nitrogen level is low. Check nitrogen requirement.")

    elif row["Nitrogen"] > 100:
        advice.append("Nitrogen level is high. Avoid unnecessary nitrogen application.")

    else:
        advice.append("Nitrogen level does not show a low/high condition.")


    # Phosphorus
    if row["Phosphorus"] < 30:
        advice.append("Phosphorus level is low. Check phosphorus requirement.")


    # Potassium
    if row["Potassium"] < 30:
        advice.append("Potassium level is low. Check potassium requirement.")


    # Rainfall
    if row["Rainfall"] < 500:
        advice.append("Rainfall is low. Additional irrigation may be required.")

    elif row["Rainfall"] > 1200:
        advice.append("Rainfall is high. Monitor waterlogging and drainage.")

    else:
        advice.append("Rainfall condition is normal based on the selected threshold.")


    # Temperature
    if row["Temperature"] < 20:
        advice.append("Temperature is low. Monitor crop growth.")

    elif row["Temperature"] > 35:
        advice.append("Temperature is high. Monitor heat stress and irrigation.")


    # Humidity
    if row["Humidity"] > 80:
        advice.append("High humidity may increase fungal disease risk.")

    elif row["Humidity"] < 40:
        advice.append("Low humidity may increase crop water requirement.")


    return status, advice


# ============================================================
# Step 9: Select Different REAL Farming Cases
# ============================================================

# Select 10 different real rows from the actual dataset
number_of_cases = min(10, len(df))

farming_cases = df.sample(
    n=number_of_cases,
    random_state=42
).reset_index(drop=True)


# ============================================================
# Step 10: Validate Each Farming Case
# ============================================================

results = []

print("\n")
print("============================================================")
print("VALIDATION OF DIFFERENT FARMING CASES")
print("============================================================")


for i in range(len(farming_cases)):

    row = farming_cases.iloc[i]

    farm_input = pd.DataFrame([{
        "Crop": row["Crop"],
        "Soil_Type": row["Soil_Type"],
        "Soil_pH": row["Soil_pH"],
        "Nitrogen": row["Nitrogen"],
        "Phosphorus": row["Phosphorus"],
        "Potassium": row["Potassium"],
        "Temperature": row["Temperature"],
        "Rainfall": row["Rainfall"],
        "Humidity": row["Humidity"]
    }])

    predicted_yield = model.predict(farm_input)[0]

    status, advice = get_advisory(
        row,
        predicted_yield
    )

    difference = abs(
        row["Yield"] - predicted_yield
    )

    results.append({
        "Case": i + 1,
        "Crop": row["Crop"],
        "Actual Yield": round(row["Yield"], 2),
        "Predicted Yield": round(predicted_yield, 2),
        "Difference": round(difference, 2),
        "Yield Status": status,
        "Advice Count": len(advice)
    })


# ============================================================
# Step 11: Display Validation Results
# ============================================================

results_df = pd.DataFrame(results)

print("\n========== VALIDATION RESULTS ==========")

display(results_df)


# ============================================================
# Step 12: Show Detailed Advisory for Each Case
# ============================================================

print("\n")
print("============================================================")
print("DETAILED FARMER ADVISORY")
print("============================================================")


for i in range(len(farming_cases)):

    row = farming_cases.iloc[i]

    farm_input = pd.DataFrame([{
        "Crop": row["Crop"],
        "Soil_Type": row["Soil_Type"],
        "Soil_pH": row["Soil_pH"],
        "Nitrogen": row["Nitrogen"],
        "Phosphorus": row["Phosphorus"],
        "Potassium": row["Potassium"],
        "Temperature": row["Temperature"],
        "Rainfall": row["Rainfall"],
        "Humidity": row["Humidity"]
    }])

    predicted_yield = model.predict(farm_input)[0]

    status, advice = get_advisory(
        row,
        predicted_yield
    )

    print("\n------------------------------")
    print("Farm Case:", i + 1)
    print("Crop:", row["Crop"])
    print("Soil Type:", row["Soil_Type"])
    print("Actual Yield:", round(row["Yield"], 2))
    print("Predicted Yield:", round(predicted_yield, 2))
    print("Yield Status:", status)

    print("\nAdvisory:")

    for item in advice:
        print("-", item)


# ============================================================
# Step 13: Calculate Validation Statistics
# ============================================================

average_difference = results_df["Difference"].mean()

high_cases = len(
    results_df[results_df["Yield Status"] == "High"]
)

medium_cases = len(
    results_df[results_df["Yield Status"] == "Medium"]
)

low_cases = len(
    results_df[results_df["Yield Status"] == "Low"]
)


print("\n")
print("============================================================")
print("VALIDATION SUMMARY")
print("============================================================")

print("Number of farming cases tested:", len(results_df))

print(
    "Average prediction difference:",
    round(average_difference, 2)
)

print("High yield cases:", high_cases)
print("Medium yield cases:", medium_cases)
print("Low yield cases:", low_cases)


# ============================================================
# Step 14: Final Validation Message
# ============================================================

print("\n============================================================")
print("FINAL RESULT")
print("============================================================")

print("✓ Actual dataset used")
print("✓ Different real farming cases tested")
print("✓ Yield forecasting tested")
print("✓ Soil conditions checked")
print("✓ Nutrient conditions checked")
print("✓ Rainfall conditions checked")
print("✓ Temperature conditions checked")
print("✓ Humidity conditions checked")
print("✓ Farmer advisory generated")

print("\nValidation completed successfully!")

Dataset loaded successfully!
Dataset shape: (600, 10)

Available Crops:
['Millets' 'Sugarcane' 'Maize' 'Cotton' 'Rice' 'Wheat' 'Groundnut']

Model trained successfully!

========== MODEL ACCURACY ==========
MAE  : 1.03
RMSE : 2.47
R2   : 0.99


VALIDATION OF DIFFERENT FARMING CASES

========== VALIDATION RESULTS ==========


,Case,Crop,Actual Yield,Predicted Yield,Difference,Yield Status,Advice Count
0,1,Cotton,2.00,2.11,0.11,Low,4
1,2,Sugarcane,68.22,69.70,1.48,High,4
2,3,Sugarcane,55.09,66.97,11.88,High,5
3,4,Maize,3.71,2.77,0.94,Low,4
4,5,Rice,4.36,4.05,0.31,Low,4
5,6,Rice,4.64,4.47,0.17,Low,5
6,7,Sugarcane,60.92,64.64,3.72,High,4
7,8,Cotton,1.78,1.73,0.05,Low,4
8,9,Wheat,3.69,3.62,0.07,Low,4
9,10,Wheat,3.24,3.29,0.05,Low,5




DETAILED FARMER ADVISORY

------------------------------
Farm Case: 1
Crop: Cotton
Soil Type: Loamy
Actual Yield: 2.0
Predicted Yield: 2.11
Yield Status: Low

Advisory:
- Expected yield is low. Crop management should be monitored.
- Soil pH is within the selected normal range.
- Nitrogen level does not show a low/high condition.
- Rainfall condition is normal based on the selected threshold.

------------------------------
Farm Case: 2
Crop: Sugarcane
Soil Type: Black
Actual Yield: 68.22
Predicted Yield: 69.7
Yield Status: High

Advisory:
- Expected yield is high.
- Soil pH is within the selected normal range.
- Nitrogen level does not show a low/high condition.
- Rainfall is high. Monitor waterlogging and drainage.

------------------------------
Farm Case: 3
Crop: Sugarcane
Soil Type: Loamy
Actual Yield: 55.09
Predicted Yield: 66.97
Yield Status: High

Advisory:
- Expected yield is high.
- Soil pH is within the selected normal range.
- Nitrogen level does not show a low/high condit